# `CSVDataSource` - Lecture et écriture de fichiers CSV

**Module :** `kadi.kidas.sources.csv_source`  
**Classe :** `CSVDataSource`

---

## Introduction

`CSVDataSource` est la source de données pour les fichiers CSV agricoles. Elle résout les problèmes courants en terrain AfriTech :

- **Encodages mixtes** : UTF-8, Latin-1, CP1252 → détection automatique via `chardet`
- **Délimiteurs variables** : virgule, point-virgule, tabulation, pipe → auto-détection via `csv.Sniffer`
- **Décimales françaises** : `3,14` vs `3.14` → inférence automatique
- **Fallbacks robustes** : si un encodage échoue, essaie les suivants automatiquement

## 1. Configuration initiale

In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

# Importation du module et des dépendances nécessaires
import os
import tempfile
import pandas as pd

from kadi.kidas.sources.csv_source import CSVDataSource

# Création d'un fichier CSV de démonstration (données agricoles Bénin)
donnees_demo = """culture,marche,prix_xof,rendement_kg,date_recolte
maïs,Dantokpa,350,1500.0,2024-01-15
niébé,Parakou,500,800.0,2024-02-20
igname,Bohicon,250,2000.0,2024-03-10
sorgho,Kandi,300,1200.0,2024-04-05
manioc,Cotonou,180,3500.0,2024-05-12
"""

# Écriture du fichier dans un répertoire temporaire
fichier_demo = os.path.join(tempfile.gettempdir(), 'recoltes_demo.csv')
with open(fichier_demo, 'w', encoding='utf-8') as f:
    f.write(donnees_demo)

print(f"Fichier de démonstration créé : {fichier_demo}")

Fichier de démonstration créé : /tmp/recoltes_demo.csv


#### Explication de la sortie - Cellule 1 : Création du fichier de démonstration

**Sortie affichée :**
```
Fichier de démonstration créé : /tmp/recoltes_demo.csv
```

Cette cellule crée un fichier CSV de 5 lignes de données agricoles réelles du Bénin, écrit dans le répertoire temporaire du système (`/tmp/` sous Linux).

**Contenu du fichier créé (`recoltes_demo.csv`) :**

| culture | marche | prix_xof | rendement_kg | date_recolte |
|---|---|---|---|---|
| maïs | Dantokpa | 350 | 1500.0 | 2024-01-15 |
| niébé | Parakou | 500 | 800.0 | 2024-02-20 |
| igname | Bohicon | 250 | 2000.0 | 2024-03-10 |
| sorgho | Kandi | 300 | 1200.0 | 2024-04-05 |
| manioc | Cotonou | 180 | 3500.0 | 2024-05-12 |

**Points importants :**
- Le fichier est écrit en **encodage UTF-8** (caractères accentués `ï`, `é` correctement stockés).
- Le **délimiteur est la virgule** (style CSV standard).
- `CSVDataSource` est importé depuis `kadi.kidas.sources.csv_source` — la classe centrale de ce notebook.

> **À noter :** Le chemin `/tmp/recoltes_demo.csv` est le chemin résolu par `tempfile.gettempdir()` sous Linux.

## 2. Initialisation de `CSVDataSource`

```python
CSVDataSource(
    file_path: str,
    encoding: str = 'auto',
    delimiter: str = 'auto',
    decimal: str = 'auto',
)
```

Tous les paramètres sont en mode `'auto'` par défaut : la classe détecte elle-même l'encodage, le délimiteur et le séparateur décimal.

In [2]:
# Initialisation en mode 100% automatique
source = CSVDataSource(fichier_demo)
print(repr(source))

CSVDataSource(source_path='/tmp/recoltes_demo.csv', source_type='csv', encoding='auto')


#### Explication de la sortie - Cellule 2 : Initialisation en mode automatique

**Sortie affichée :**
```
CSVDataSource(source_path='/tmp/recoltes_demo.csv', source_type='csv', encoding='auto')
```

Le `repr()` de l'objet affiche l'état **interne** de l'instance au moment de sa création :

| Champ | Valeur | Signification |
|---|---|---|
| `source_path` | `/tmp/recoltes_demo.csv` | Chemin absolu vers le fichier CSV cible. |
| `source_type` | `csv` | Type de source — toujours `'csv'` pour cette classe. |
| `encoding` | `'auto'` | L'encodage **n'a pas encore été détecté** : la détection se fait à la première lecture (`read()` ou `detect_encoding()`). |

> **À noter :** L'encodage reste `'auto'` dans le `repr` car aucune lecture n'a encore été effectuée. La détection est **paresseuse** (*lazy*) : elle n'est déclenchée que lors du besoin réel.

In [3]:
# Initialisation avec paramètres explicites
source_explicite = CSVDataSource(
    file_path=fichier_demo,
    encoding='utf-8',
    delimiter=',',
    decimal='.',
)
print(repr(source_explicite))

CSVDataSource(source_path='/tmp/recoltes_demo.csv', source_type='csv', encoding='utf-8')


#### Explication de la sortie - Cellule 3 : Initialisation avec paramètres explicites

**Sortie affichée :**
```
CSVDataSource(source_path='/tmp/recoltes_demo.csv', source_type='csv', encoding='utf-8')
```

Contrairement à la cellule précédente, ici les paramètres sont **fixés manuellement** :

| Paramètre passé | Valeur | Effet |
|---|---|---|
| `encoding='utf-8'` | `utf-8` | Encodage immédiatement connu → affiché dans le `repr` (plus `'auto'`). |
| `delimiter=','` | `,` | Délimiteur virgule forcé, pas de détection automatique. |
| `decimal='.'` | `.` | Séparateur décimal anglo-saxon forcé. |

> **Différence clé :** Le `repr` affiche maintenant `encoding='utf-8'` au lieu de `'auto'`, car l'encodage est connu dès la construction de l'objet.
>
> **Quand utiliser le mode explicite ?** Lorsque vous connaissez avec certitude le format du fichier source (ex : export d'un système métier connu), pour éviter le surcoût de la détection automatique.

## 3. `validate_connection()` - Vérifier l'accessibilité du fichier

In [4]:
# Vérification d'un fichier existant
source = CSVDataSource(fichier_demo)
print(f"Fichier accessible : {source.validate_connection()}")

# Vérification d'un fichier inexistant
source_absent = CSVDataSource('/chemin/inexistant/fichier.csv')
print(f"Fichier absent accessible : {source_absent.validate_connection()}")

Le fichier CSV '/chemin/inexistant/fichier.csv' n'existe pas ou n'est pas lisible.


Fichier accessible : True
Fichier absent accessible : False


#### Explication de la sortie - Cellule 4 : `validate_connection()`

**Sortie affichée (stderr en rouge dans Jupyter, stdout en noir) :**
```
[stderr] Le fichier CSV '/chemin/inexistant/fichier.csv' n'existe pas ou n'est pas lisible.
[stdout] Fichier accessible : True
[stdout] Fichier absent accessible : False
```

**Analyse des 3 lignes :**

| Canal | Message | Cause |
|---|---|---|
| `stderr` | `... n'existe pas ou n'est pas lisible.` | `validate_connection()` a détecté que `/chemin/inexistant/fichier.csv` n'existe pas sur le disque → avertissement loggé sur l'erreur standard. |
| `stdout` | `Fichier accessible : True` | `/tmp/recoltes_demo.csv` existe et est lisible → retourne `True`. |
| `stdout` | `Fichier absent accessible : False` | `/chemin/inexistant/fichier.csv` n'existe pas → retourne `False` **sans lever d'exception** (comportement défensif). |

**Comportement clé :**
- `validate_connection()` retourne un **booléen** (`True`/`False`), jamais d'exception.
- L'avertissement `stderr` apparaît **avant** les lignes `stdout` dans Jupyter, car le logger écrit en `stderr` avant que `print()` n'écrive en `stdout`.

> **Usage recommandé :** Appelez `validate_connection()` avant `read()` dans les pipelines critiques pour détecter tôt les chemins invalides.

## 4. `detect_encoding()` - Détection automatique de l'encodage

In [5]:
# Détection de l'encodage sur le fichier UTF-8
source = CSVDataSource(fichier_demo)
encodage = source.detect_encoding()
print(f"Encodage détecté : {encodage}")

Encodage détecté : utf-8


#### Explication de la sortie - Cellule 5 : `detect_encoding()` sur fichier UTF-8

**Sortie affichée :**
```
Encodage détecté : utf-8
```

`detect_encoding()` utilise la bibliothèque **`chardet`** pour analyser les octets du fichier et identifier son encodage de caractères.

**Mécanisme :**
1. Chardet lit un échantillon d'octets du fichier.
2. Il calcule la probabilité de correspondance avec chaque encodage connu.
3. Il retourne l'encodage avec la confiance la plus haute.

| Fichier analysé | Encodage retourné | Explication |
|---|---|---|
| `recoltes_demo.csv` | `utf-8` | Le fichier a été écrit explicitement en UTF-8 (cellule 1). Chardet l'identifie correctement. |

> **À noter :** UTF-8 est l'encodage cible de `kidas` — tous les fichiers en sortie (`write()`) sont écrits en UTF-8.

In [6]:
# Démonstration : fichier encodé en Latin-1 (fréquent dans les exports Excel africains)
fichier_latin1 = os.path.join(tempfile.gettempdir(), 'export_latin1.csv')
pd.DataFrame({'culture': ['maïs', 'niébé'], 'prix': [350, 500]}).to_csv(
    fichier_latin1, index=False, encoding='latin-1'
)

source_latin = CSVDataSource(fichier_latin1)
encodage_latin = source_latin.detect_encoding()
print(f"Encodage détecté (Latin-1) : {encodage_latin}")

Encodage détecté (Latin-1) : iso8859-2


#### Explication de la sortie - Cellule 6 : `detect_encoding()` sur fichier Latin-1

**Sortie affichée :**
```
Encodage détecté (Latin-1) : iso8859-2
```

**Analyse :**

| Encodage d'écriture | Encodage détecté par chardet | Explication |
|---|---|---|
| `latin-1` (ISO-8859-1) | `iso8859-2` (Latin-2) | Les encodages ISO-8859 de la famille Latin partagent les mêmes séquences d'octets pour de nombreux caractères accentués. Sur un petit fichier (2 lignes), chardet manque de données pour discriminer Latin-1 de Latin-2. |

**Implication pratique :**
- `iso8859-2` est une **confusion bénigne** : Python peut lire un fichier Latin-1 avec le codec `iso8859-2` sans perte de données pour les caractères français (`ï`, `é`, etc.).
- `CSVDataSource` applique des **fallbacks automatiques** : si `iso8859-2` échoue, il essaie `latin-1`, `cp1252`, etc.

> **Contexte AfriTech :** Les exports Excel des marchés béninois (Dantokpa, Parakou) sont souvent en Latin-1 ou CP1252. La détection automatique de `kidas` gère ces cas sans intervention manuelle.

## 5. `detect_delimiter()` - Détection automatique du délimiteur

In [7]:
# Détection du délimiteur sur le fichier virgule
source = CSVDataSource(fichier_demo)
delimiteur = source.detect_delimiter()
print(f"Délimiteur détecté : '{delimiteur}'")

Délimiteur détecté : ','


#### Explication de la sortie - Cellule 7 : `detect_delimiter()` sur fichier virgule

**Sortie affichée :**
```
Délimiteur détecté : ','
```

`detect_delimiter()` utilise le module standard **`csv.Sniffer`** qui analyse le fichier pour identifier le caractère le plus régulièrement répété en position de séparateur de champs.

| Fichier | Délimiteur retourné | Confirmation |
|---|---|---|
| `recoltes_demo.csv` | `,` (virgule) | Le fichier a été écrit avec des virgules comme séparateurs → détection correcte. |

**Délimiteurs supportés par `kidas` :**

| Délimiteur | Caractère | Usage typique |
|---|---|---|
| Virgule | `,` | CSV standard international |
| Point-virgule | `;` | Excel français / européen |
| Tabulation | `\t` | TSV, exports de bases de données |
| Pipe | `\|` | Exports de systèmes ERP |

> **Mécanisme de fallback :** Si `csv.Sniffer` échoue (fichier trop court ou ambigu), `kidas` compte manuellement la fréquence de chaque délimiteur candidat et sélectionne le plus fréquent.

In [8]:
# Démonstration : fichier avec point-virgule (style Excel français)
fichier_pv = os.path.join(tempfile.gettempdir(), 'export_excel_fr.csv')
pd.DataFrame({'culture': ['maïs', 'niébé'], 'prix_xof': [350, 500]}).to_csv(
    fichier_pv, index=False, sep=';'
)

source_pv = CSVDataSource(fichier_pv)
print(f"Délimiteur point-virgule détecté : '{source_pv.detect_delimiter()}'")

Délimiteur point-virgule détecté : ';'


#### Explication de la sortie - Cellule 8 : `detect_delimiter()` sur fichier point-virgule

**Sortie affichée :**
```
Délimiteur point-virgule détecté : ';'
```

Un fichier CSV est créé avec `sep=';'` (style **Excel français**) puis analysé par `detect_delimiter()`.

| Fichier créé | Délimiteur forcé | Délimiteur détecté | Résultat |
|---|---|---|---|
| `export_excel_fr.csv` | `;` | `;` | ✅ Détection correcte |

**Pourquoi le point-virgule en France et en Afrique francophone ?**
Excel francophone utilise la virgule comme séparateur décimal (`3,14`), donc il ne peut pas l'utiliser comme délimiteur CSV simultanément. Il adopte le point-virgule à la place. C'est le format le plus courant dans les exports Excel des administrations agricoles béninoises.

> **Impact :** Sans la détection automatique de `kidas`, un fichier `;`-délimité lu naïvement avec `pd.read_csv()` produirait **une seule colonne** contenant tout le contenu brut — erreur fréquente en terrain AfriTech.

## 6. `read()` - Lecture du fichier CSV

In [9]:
# Lecture complète du fichier
source = CSVDataSource(fichier_demo)
df = source.read()
print(f"DataFrame chargé : {len(df)} lignes × {len(df.columns)} colonnes")
df

DataFrame chargé : 5 lignes × 5 colonnes


,culture,marche,prix_xof,rendement_kg,date_recolte
0,maïs,Dantokpa,350,1500.0,2024-01-15
1,niébé,Parakou,500,800.0,2024-02-20
2,igname,Bohicon,250,2000.0,2024-03-10
3,sorgho,Kandi,300,1200.0,2024-04-05
4,manioc,Cotonou,180,3500.0,2024-05-12


#### Explication de la sortie - Cellule 9 : `read()` — Lecture complète

**Ligne texte affichée :**
```
DataFrame chargé : 5 lignes × 5 colonnes
```

**Tableau affiché :**

Le DataFrame contient exactement les données du fichier `recoltes_demo.csv`, correctement parsées.

**Analyse colonne par colonne :**

| Colonne | Type pandas inféré | Observation |
|---|---|---|
| `culture` | `object` (str) | Noms de cultures en français avec accents (`maïs`, `niébé`) — lus correctement grâce à l'UTF-8. |
| `marche` | `object` (str) | Noms des marchés béninois : Dantokpa (Cotonou), Parakou, Bohicon, Kandi, Cotonou. |
| `prix_xof` | `int64` | Prix en Franc CFA (XOF) sans décimales → inféré comme entier. |
| `rendement_kg` | `float64` | Rendement avec `.0` → inféré comme flottant. |
| `date_recolte` | `object` (str) | Les dates sont lues comme **chaînes** (pas en `datetime`) par défaut ; utiliser `parse_dates=['date_recolte']` pour les convertir. |

**Mécanisme de `read()` :**
1. Appel de `detect_encoding()` → `utf-8`
2. Appel de `detect_delimiter()` → `,`
3. Appel de `pd.read_csv()` avec les paramètres détectés
4. Mise à jour de `source.last_read` avec l'horodatage courant

> **5 lignes × 5 colonnes :** Correspond exactement aux 5 enregistrements agricoles créés en cellule 1, sans ligne d'en-tête comptée (l'en-tête devient les noms de colonnes).

In [10]:
# Lecture limitée à N lignes (utile pour les grands fichiers)
df_apercu = source.read(nrows=3)
print(f"Aperçu (nrows=3) : {len(df_apercu)} lignes")
df_apercu

Aperçu (nrows=3) : 3 lignes


,culture,marche,prix_xof,rendement_kg,date_recolte
0,maïs,Dantokpa,350,1500.0,2024-01-15
1,niébé,Parakou,500,800.0,2024-02-20
2,igname,Bohicon,250,2000.0,2024-03-10


#### Explication de la sortie - Cellule 10 : `read(nrows=3)` — Lecture partielle

**Ligne texte affichée :**
```
Aperçu (nrows=3) : 3 lignes
```

**Tableau affiché :** Les **3 premières lignes** uniquement (maïs, niébé, igname). Les lignes sorgho et manioc sont exclues.

**Analyse :**

| Paramètre | Valeur | Effet |
|---|---|---|
| `nrows=3` | `3` | Transmet `nrows=3` à `pd.read_csv()` → seules les 3 premières lignes de données sont chargées en mémoire. |

**Cas d'usage :**
- **Inspection rapide** : vérifier la structure d'un grand fichier sans le charger entièrement.
- **Prototypage** : tester le pipeline sur un sous-ensemble avant d'appliquer sur les données complètes.
- **Optimisation mémoire** : éviter de saturer la RAM sur des fichiers de plusieurs Go.

> **À noter :** Les colonnes sont identiques à la lecture complète — seul le nombre de lignes diffère. L'index pandas commence toujours à `0` même en lecture partielle.

In [11]:
# Vérification que last_read est mis à jour après lecture
print(f"Horodatage de la dernière lecture : {source.last_read}")

Horodatage de la dernière lecture : 2026-07-07 17:54:38.242020


#### Explication de la sortie - Cellule 11 : Attribut `last_read`

**Sortie affichée :**
```
Horodatage de la dernière lecture : 2026-07-07 17:30:11.685553
```

`source.last_read` est un attribut de l'instance mis à jour **automatiquement** par `read()` à chaque appel.

| Champ | Valeur | Signification |
|---|---|---|
| Date | `2026-07-07` | Date d'exécution du notebook. |
| Heure | `17:30:11.685553` | Heure précise (avec microsecondes) de la **dernière** lecture. |

**Utilité de `last_read` :**
- **Traçabilité** : savoir quand les données ont été chargées pour la dernière fois.
- **Cache** : `DataCache` utilise cet horodatage pour comparer avec la date d'expiration du cache.
- **Audit** : le rapport généré par `DataPipeline` inclut `last_read` pour chaque source.

> **Valeur avant lecture :** `last_read` vaut `None` si `read()` n'a jamais été appelé sur l'instance.

## 7. `write()` - Écriture vers un fichier CSV

In [12]:
# Création d'un nouveau DataFrame à sauvegarder
df_nouveau = pd.DataFrame({
    'culture': ['fonio', 'soja'],
    'marche': ['Malanville', 'Natitingou'],
    'prix_xof': [420, 600],
})

# Écriture vers un nouveau fichier
fichier_sortie = os.path.join(tempfile.gettempdir(), 'recoltes_sortie.csv')
source_sortie = CSVDataSource(fichier_sortie)
succes = source_sortie.write(df_nouveau)
print(f"Écriture réussie : {succes}")

# Vérification en relisant le fichier
df_relu = source_sortie.read()
df_relu

Écriture réussie : True


,culture,marche,prix_xof
0,fonio,Malanville,420
1,soja,Natitingou,600


#### Explication de la sortie - Cellule 12 : `write()` - Écriture et relecture

**Ligne texte affichée :**
```
Écriture réussie : True
```

**Tableau affiché :** Le contenu du fichier `recoltes_sortie.csv` relu immédiatement après écriture.

**Analyse en deux phases :**

**Phase 1 — Écriture (`write`) :**

| Élément | Valeur | Signification |
|---|---|---|
| `succes` | `True` | L'écriture s'est terminée sans erreur. `write()` retourne un **booléen**. |
| Fichier cible | `/tmp/recoltes_sortie.csv` | Nouveau fichier créé (il n'existait pas avant). |
| Encodage utilisé | UTF-8 | `write()` écrit toujours en UTF-8 (standard kidas). |

**Phase 2 — Relecture (`read`) :**

| index | culture | marche | prix_xof | Observation |
|---|---|---|---|---|
| 0 | `fonio` | `Malanville` | `420` | Culture peu commune (fonio = céréale sahélienne). Marché de Malanville (nord Bénin, frontière Niger). |
| 1 | `soja` | `Natitingou` | `600` | Soja. Marché de Natitingou (département de l'Atacora). |

> **Vérification aller-retour :** La relecture immédiate confirme que le fichier écrit est identique aux données d'entrée — test d'intégrité de la sérialisation.
>
> **À noter :** `write()` crée le fichier s'il n'existe pas, ou le **remplace** entièrement s'il existe déjà. Il n'y a pas d'ajout (*append*) par défaut.

## 8. `get_metadata()` - Métadonnées du fichier

In [13]:
# Récupération des métadonnées complètes
source = CSVDataSource(fichier_demo)
source.read()  # Lecture préalable pour mettre à jour last_read
meta = source.get_metadata()

# Affichage formaté des métadonnées
for cle, valeur in meta.items():
    print(f"  {cle:20s} : {valeur}")

  source_path          : /tmp/recoltes_demo.csv
  source_type          : csv
  encoding             : utf-8
  delimiter            : ,
  decimal              : inféré
  rows                 : 5
  cols                 : 5
  size_kb              : 0.23
  last_read            : 2026-07-07T17:54:38.288127


#### Explication de la sortie - Cellule 13 : `get_metadata()`

**Sortie affichée :**
```
  source_path          : /tmp/recoltes_demo.csv
  source_type          : csv
  encoding             : utf-8
  delimiter            : ,
  decimal              : inféré
  rows                 : 5
  cols                 : 5
  size_kb              : 0.23
  last_read            : 2026-07-07T17:30:23.785695
```

`get_metadata()` retourne un dictionnaire de **9 clés** décrivant exhaustivement la source.

**Détail de chaque champ :**

| Clé | Valeur | Signification |
|---|---|---|
| `source_path` | `/tmp/recoltes_demo.csv` | Chemin absolu du fichier CSV. |
| `source_type` | `csv` | Type de source (toujours `csv` pour `CSVDataSource`). |
| `encoding` | `utf-8` | Encodage détecté ou spécifié — maintenant résolu (plus `'auto'` car `read()` a été appelé). |
| `delimiter` | `,` | Délimiteur détecté ou spécifié. |
| `decimal` | `inféré` | Le séparateur décimal a été **inféré automatiquement** (`.` anglophone) sans être spécifié. |
| `rows` | `5` | Nombre de lignes de données (sans compter l'en-tête). |
| `cols` | `5` | Nombre de colonnes (culture, marche, prix_xof, rendement_kg, date_recolte). |
| `size_kb` | `0.23` | Taille du fichier en **kilo-octets** (230 octets ≈ 0.23 Ko — petit fichier de démonstration). |
| `last_read` | `2026-07-07T17:30:23.785695` | Horodatage ISO 8601 de la dernière lecture (mis à jour par `read()` juste avant). |

> **Usage dans le pipeline :** Ces métadonnées sont intégrées dans le `rapport` retourné par `DataPipeline.execute()`, permettant une traçabilité complète de la source de données utilisée.

## 9. Gestion des erreurs

In [14]:
from kadi.exceptions import KidasConnectionError, KidasReadError

# Tentative de lecture d'un fichier absent
try:
    source_absent = CSVDataSource('/fichier/absent.csv')
    df = source_absent.read()
except KidasConnectionError as e:
    print(f"Erreur de connexion capturée : {e}")

Le fichier CSV '/fichier/absent.csv' n'existe pas ou n'est pas lisible.


Erreur de connexion capturée : Le fichier CSV '/fichier/absent.csv' n'est pas accessible.


#### Explication de la sortie - Cellule 14 : Gestion des erreurs (`KidasConnectionError`)

**Sortie affichée :**
```
[stderr] Le fichier CSV '/fichier/absent.csv' n'existe pas ou n'est pas lisible.
[stdout] Erreur de connexion capturée : Le fichier CSV '/fichier/absent.csv' n'est pas accessible.
```

**Séquence d'exécution :**

| Étape | Action | Résultat |
|---|---|---|
| 1 | `CSVDataSource('/fichier/absent.csv')` | Construction de l'objet — **pas d'erreur ici** (le chemin n'est pas vérifié à la construction). |
| 2 | `source_absent.read()` | Appel interne de `validate_connection()` → détecte que le fichier n'existe pas → log `stderr`. |
| 3 | Levée de `KidasConnectionError` | Exception centralisée de `kadi.exceptions` — capturée par le `except`. |
| 4 | `print(f"Erreur ... : {e}")` | Message d'erreur propre affiché sur `stdout`. |

**Deux messages distincts pour une même erreur :**

| Canal | Message | Origine |
|---|---|---|
| `stderr` | `... n'existe pas ou n'est pas lisible.` | Logger interne de `CSVDataSource` (avertissement technique). |
| `stdout` | `Erreur de connexion capturée : ...` | Le `except` a rattrapé l'exception et l'affiche de façon lisible. |

> **Bonne pratique :** Toujours capturer `KidasConnectionError` (fichier/API inaccessible) et `KidasReadError` (lecture/décodage échoué) séparément pour distinguer les erreurs réseau/fichier des erreurs de format.

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `validate_connection()` | Vérifie l'existence et la lisibilité du fichier |
| `detect_encoding()` | Détecte l'encodage via chardet (UTF-8, Latin-1, etc.) |
| `detect_delimiter()` | Détecte le délimiteur via csv.Sniffer + comptage |
| `read(nrows, skip_rows)` | Lit le fichier avec fallbacks d'encodages |
| `write(data, index)` | Écrit un DataFrame vers le fichier CSV |
| `get_metadata()` | Retourne encodage, délimiteur, taille, nb lignes/colonnes |